In [27]:
words = open('names.txt', 'r').read().splitlines()
b = {}
for word in words:
  word_pre_suf = ['<'] + list(word) + [']']
  for i in range(len(word_pre_suf) - 2):
    bigram = word_pre_suf[i] + word_pre_suf[i + 1]
    b[bigram] = b.get(bigram, 0) + 1

b = sorted(b.items(), key=lambda x: x[1], reverse=True)
print(b)

[('an', 5438), ('<a', 4410), ('ar', 3264), ('el', 3248), ('ri', 3033), ('na', 2977), ('<k', 2963), ('le', 2921), ('en', 2675), ('la', 2623), ('ma', 2590), ('<m', 2538), ('al', 2528), ('li', 2480), ('ia', 2445), ('<j', 2422), ('on', 2411), ('ra', 2356), ('ah', 2332), ('ha', 2244), ('ya', 2143), ('in', 2126), ('<s', 2055), ('ay', 2050), ('er', 1958), ('nn', 1906), ('yn', 1826), ('ka', 1731), ('ni', 1725), ('re', 1697), ('<d', 1690), ('ie', 1653), ('ai', 1650), ('<r', 1639), ('am', 1634), ('ly', 1588), ('<l', 1572), ('<c', 1542), ('<e', 1531), ('ja', 1473), ('ne', 1359), ('ll', 1345), ('il', 1345), ('is', 1316), ('<t', 1308), ('<b', 1306), ('da', 1303), ('sh', 1285), ('de', 1283), ('ee', 1271), ('mi', 1256), ('sa', 1201), ('<n', 1146), ('as', 1118), ('yl', 1104), ('ey', 1070), ('or', 1059), ('ad', 1042), ('ta', 1027), ('<z', 929), ('vi', 911), ('ke', 895), ('se', 884), ('<h', 874), ('ro', 869), ('es', 861), ('za', 860), ('ir', 849), ('br', 842), ('av', 834), ('me', 818), ('ei', 818), ('ca

In [28]:
import torch
N = torch.zeros((27, 27), dtype=torch.int32)

sorted_chars = sorted(list(set(''.join(words))))
enumerated = enumerate(sorted_chars)
char_to_idx = {char: idx + 1 for idx, char in enumerated}
char_to_idx['*'] = 0
idx_to_char = {idx: char for char, idx in char_to_idx.items()}


for word in words:
  word = '*' + word + '*'
  for ch1, ch2 in zip(word, word[1:]):
    ix1 = char_to_idx[ch1]
    ix2 = char_to_idx[ch2]
    N[ix1, ix2] += 1


In [ ]:
P = N.float()
P = (P + 1) / P.sum(1, keepdims=True)

In [44]:
# Creating and training a neural net
import torch.nn.functional as F
import matplotlib.pyplot as plt

W = torch.randn((27, 27), generator=torch.Generator().manual_seed(2147481317), requires_grad=True)

def get_datasets(train_pct, dev_pct, test_pct):
  train_size = int(train_pct * len(words))
  dev_size = int(dev_pct * len(words))
  test_size = int(test_pct * len(words))
  train_words = words[:train_size]
  dev_words = words[train_size:train_size + dev_size]
  test_words = words[train_size + dev_size:train_size + dev_size + test_size]
  return train_words, dev_words, test_words

train_words, dev_words, test_words = get_datasets(0.8, 0.1, 0.1)

def get_bigram_dataset(words):
  xs = []
  ys = []
  for word in words:
    word = '*' + word + '*'
    for ch1, ch2 in zip(word, word[1:]):
      ix1 = char_to_idx[ch1]
      ix2 = char_to_idx[ch2]
      xs.append(ix1)
      ys.append(ix2)
  return xs, ys

train_xs, train_ys = get_bigram_dataset(train_words)
dev_xs, dev_ys = get_bigram_dataset(dev_words)
test_xs, test_ys = get_bigram_dataset(test_words)

train_xs = torch.tensor(train_xs)
train_ys = torch.tensor(train_ys)

dev_xs = torch.tensor(dev_xs)
dev_ys = torch.tensor(dev_ys)

test_xs = torch.tensor(test_xs)
test_ys = torch.tensor(test_ys)


train_xs_encoded = F.one_hot(train_xs, num_classes=27).float()
print(train_xs_encoded.shape)
print(W.shape)

print(len(words))
print(len(train_xs))
print(len(dev_xs))
print(len(test_xs))

torch.Size([182778, 27])
torch.Size([27, 27])
32033
182778
22633
22729


In [63]:
for i in range(500):
  ## Forward pass
  matmul = train_xs_encoded @ W

  counts = matmul.exp()
  probs = counts /counts.sum(1, keepdim=True)

  arr = torch.arange(len(train_xs))
  vectorised_loss = -(probs[arr, train_ys]).log().mean() + 0.003 * (W**2).mean()
  print(f"Loss {vectorised_loss}")

  ## Cross entropy version
  # cross_entropy_loss = F.cross_entropy(x_encoded @ W, ys)
  # print(f"Loss {cross_entropy_loss}")

  # W.grad = None
  # cross_entropy_loss.backward()

  ## Backward pass
  W.grad = None
  vectorised_loss.backward()

  W.data += -5 * W.grad


Loss 2.554750919342041
Loss 2.545067310333252
Loss 2.536358118057251
Loss 2.528533935546875
Loss 2.521517038345337
Loss 2.515239715576172
Loss 2.5096399784088135
Loss 2.5046586990356445
Loss 2.500239133834839
Loss 2.496326446533203
Loss 2.4928665161132812
Loss 2.489806890487671
Loss 2.4870989322662354
Loss 2.48469614982605
Loss 2.4825570583343506
Loss 2.4806439876556396
Loss 2.478924036026001
Loss 2.4773693084716797
Loss 2.4759552478790283
Loss 2.4746615886688232
Loss 2.4734716415405273
Loss 2.4723708629608154
Loss 2.4713475704193115
Loss 2.4703919887542725
Loss 2.469496250152588
Loss 2.468654155731201
Loss 2.4678592681884766
Loss 2.4671075344085693
Loss 2.4663946628570557
Loss 2.465717315673828
Loss 2.4650728702545166
Loss 2.464458465576172
Loss 2.463871717453003
Loss 2.463310718536377
Loss 2.4627740383148193
Loss 2.462259531021118
Loss 2.461766481399536
Loss 2.4612929821014404
Loss 2.4608380794525146
Loss 2.4604008197784424
Loss 2.459980010986328
Loss 2.4595746994018555
Loss 2.459184

In [66]:
#### Test bigram model on dev dataset
for i in range(100):
  dev_xs_encoded = F.one_hot(dev_xs, num_classes=27).float()
  matmul = dev_xs_encoded @ W
  dev_counts = matmul.exp()
  dev_probs = dev_counts /dev_counts.sum(1, keepdim=True)

  arr = torch.arange(len(dev_xs))
  dev_vectorised_loss = -(dev_probs[arr, dev_ys]).log().mean() + 0.001 * (W**2).mean()
  print(f"Loss on dev dataset {dev_vectorised_loss}")

  W.grad = None
  dev_vectorised_loss.backward()

  W.data += -10 * W.grad

Loss on dev dataset 2.495384931564331
Loss on dev dataset 2.495337724685669
Loss on dev dataset 2.495291233062744
Loss on dev dataset 2.495245933532715
Loss on dev dataset 2.4952011108398438
Loss on dev dataset 2.495157480239868
Loss on dev dataset 2.495114326477051
Loss on dev dataset 2.495072364807129
Loss on dev dataset 2.4950311183929443
Loss on dev dataset 2.494990348815918
Loss on dev dataset 2.494950771331787
Loss on dev dataset 2.4949116706848145
Loss on dev dataset 2.494872808456421
Loss on dev dataset 2.494835376739502
Loss on dev dataset 2.494798421859741
Loss on dev dataset 2.4947619438171387
Loss on dev dataset 2.4947261810302734
Loss on dev dataset 2.4946913719177246
Loss on dev dataset 2.494656801223755
Loss on dev dataset 2.4946227073669434
Loss on dev dataset 2.494589328765869
Loss on dev dataset 2.494556188583374
Loss on dev dataset 2.4945242404937744
Loss on dev dataset 2.494492530822754
Loss on dev dataset 2.49446177482605
Loss on dev dataset 2.4944305419921875
Loss

In [67]:
#### Test bigram model on test dataset
test_xs_encoded = F.one_hot(test_xs, num_classes=27).float()
matmul = test_xs_encoded @ W
test_counts = matmul.exp()
test_probs = test_counts / test_counts.sum(1, keepdim=True)

arr = torch.arange(len(test_xs))
test_vectorised_loss = -(test_probs[arr, test_ys]).log().mean() + 0.001 * (W**2).mean()
print(f"Loss on test dataset {test_vectorised_loss}")

Loss on test dataset 2.5162971019744873


In [ ]:
## Generate names using bigram model
g = torch.Generator().manual_seed(2147483647)
for i in range(5):
  start_char = '*'
  ix = char_to_idx[start_char]
  out = []
  while True:
    #x_encoded = F.one_hot(torch.tensor([ix]), num_classes=27).float()
    matmul = (W[ix]).reshape(1, -1)
    #print(matmul)
    #print(matmul.shape)
    counts = matmul.exp()
    probs = counts / counts.sum(1, keepdim=True)
    ix = torch.multinomial(probs, num_samples=1, replacement=True, generator=g).item()
    out.append(idx_to_char[ix])
    if ix == 0:
      break
  print(''.join(out))

cexze*
mogh*
mikilezityha*
konimittain*
llayn*


In [68]:
## Seeding data for a trigram neural net
sorted_chars = sorted(list(set(''.join(words))))
enumerated = enumerate(sorted_chars)
char_to_idx = {char: idx + 1 for idx, char in enumerated}
char_to_idx['*'] = 0
idx_to_char = {idx: char for char, idx in char_to_idx.items()}

sorted_chars.insert(0, '*')
counter = 0
trigram_idx_map = {}
idx_trigram_map = {}
for i in range(len(sorted_chars)):
  for j in range(len(sorted_chars)):
    trigram = sorted_chars[i] + sorted_chars[j]
    trigram_idx_map[trigram] = counter
    idx_trigram_map[counter] = trigram
    counter += 1


print(sorted_chars)
print(trigram_idx_map)
print(idx_trigram_map)

['*', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']
{'**': 0, '*a': 1, '*b': 2, '*c': 3, '*d': 4, '*e': 5, '*f': 6, '*g': 7, '*h': 8, '*i': 9, '*j': 10, '*k': 11, '*l': 12, '*m': 13, '*n': 14, '*o': 15, '*p': 16, '*q': 17, '*r': 18, '*s': 19, '*t': 20, '*u': 21, '*v': 22, '*w': 23, '*x': 24, '*y': 25, '*z': 26, 'a*': 27, 'aa': 28, 'ab': 29, 'ac': 30, 'ad': 31, 'ae': 32, 'af': 33, 'ag': 34, 'ah': 35, 'ai': 36, 'aj': 37, 'ak': 38, 'al': 39, 'am': 40, 'an': 41, 'ao': 42, 'ap': 43, 'aq': 44, 'ar': 45, 'as': 46, 'at': 47, 'au': 48, 'av': 49, 'aw': 50, 'ax': 51, 'ay': 52, 'az': 53, 'b*': 54, 'ba': 55, 'bb': 56, 'bc': 57, 'bd': 58, 'be': 59, 'bf': 60, 'bg': 61, 'bh': 62, 'bi': 63, 'bj': 64, 'bk': 65, 'bl': 66, 'bm': 67, 'bn': 68, 'bo': 69, 'bp': 70, 'bq': 71, 'br': 72, 'bs': 73, 'bt': 74, 'bu': 75, 'bv': 76, 'bw': 77, 'bx': 78, 'by': 79, 'bz': 80, 'c*': 81, 'ca': 82, 'cb': 83, 'cc': 84, 'cd': 85, 'ce': 86, 'cf

In [71]:
def get_trigram_dataset(words):
  xs = []
  ys = []
  for word in words:
    work_updated = ['*', '*'] + list(word) + ['*', '*']
    for ch1, ch2, ch3 in zip(work_updated, work_updated[1:], work_updated[2:]):
      ix = trigram_idx_map[ch1 + ch2]
      iy = char_to_idx[ch3]
      xs.append(ix)
      ys.append(iy)
  return xs, ys

train_xs, train_ys = get_trigram_dataset(train_words)
dev_xs, dev_ys = get_trigram_dataset(dev_words)
test_xs, test_ys = get_trigram_dataset(test_words)
W = torch.randn((729, 27), generator=torch.Generator().manual_seed(2147481317), requires_grad=True)
print(len(train_xs), len(dev_xs), len(test_xs))
print(len(train_ys), len(dev_ys), len(test_ys))
print(W.shape)
train_xs = torch.tensor(train_xs)
train_ys = torch.tensor(train_ys)

dev_xs = torch.tensor(dev_xs)
dev_ys = torch.tensor(dev_ys)

test_xs = torch.tensor(test_xs)
test_ys = torch.tensor(test_ys)


208404 25836 25932
208404 25836 25932
torch.Size([729, 27])


In [72]:
## Gradient descent on trigram neural net (training set)
import torch.nn.functional as F
xs_encoded = F.one_hot(train_xs, num_classes=729).float()
for i in range(500):
  ## Forward pass


  # counts_arr = []
  # for x in xs:
  #   count_row = W[x]
  #   counts_arr.append(count_row)
  # counts_tensor = torch.stack(counts_arr)


  # matmul = xs_encoded @ W

  # counts = matmul.exp()
  # probs = counts /counts.sum(1, keepdim=True)

  # arr = torch.arange(len(xs))
  # vectorised_loss = -(probs[arr, ys]).log().mean() + 0.01 * (W**2).mean()
  # print(f"Loss {vectorised_loss}")

  # Cross entropy version
  cross_entropy_loss = F.cross_entropy(xs_encoded @ W, train_ys) + 0.01 * (W**2).mean()
  print(f"Training Loss {cross_entropy_loss}")

  W.grad = None
  cross_entropy_loss.backward()

  ## Backward pass
  # W.grad = None
  # vectorised_loss.backward()

  W.data += -10 * W.grad

Training Loss 3.736504077911377
Training Loss 3.6985416412353516
Training Loss 3.6620187759399414
Training Loss 3.6268322467803955
Training Loss 3.5929388999938965
Training Loss 3.560331344604492
Training Loss 3.529022216796875
Training Loss 3.499026298522949
Training Loss 3.4703450202941895
Training Loss 3.4429569244384766
Training Loss 3.4168200492858887
Training Loss 3.3918774127960205
Training Loss 3.368061065673828
Training Loss 3.345306396484375
Training Loss 3.3235557079315186
Training Loss 3.3027586936950684
Training Loss 3.2828714847564697
Training Loss 3.263857126235962
Training Loss 3.245680093765259
Training Loss 3.228304862976074
Training Loss 3.2116928100585938
Training Loss 3.1958045959472656
Training Loss 3.1805965900421143
Training Loss 3.1660256385803223
Training Loss 3.1520471572875977
Training Loss 3.1386184692382812
Training Loss 3.125699043273926
Training Loss 3.1132500171661377
Training Loss 3.1012351512908936
Training Loss 3.089622974395752
Training Loss 3.07838

In [78]:

dev_xs_encoded = F.one_hot(dev_xs, num_classes=729).float()
for i in range(100):
  ## Forward pass


  # counts_arr = []
  # for x in xs:
  #   count_row = W[x]
  #   counts_arr.append(count_row)
  # counts_tensor = torch.stack(counts_arr)


  # matmul = xs_encoded @ W

  # counts = matmul.exp()
  # probs = counts /counts.sum(1, keepdim=True)

  # arr = torch.arange(len(xs))
  # vectorised_loss = -(probs[arr, ys]).log().mean() + 0.01 * (W**2).mean()
  # print(f"Loss {vectorised_loss}")

  # Cross entropy version
  cross_entropy_loss = F.cross_entropy(dev_xs_encoded @ W, dev_ys) + 0.002 * (W**2).mean()
  print(f"Dev set Loss {cross_entropy_loss}")

  W.grad = None
  cross_entropy_loss.backward()

  ## Backward pass
  # W.grad = None
  # vectorised_loss.backward()

  W.data += -50 * W.grad

Dev set Loss 2.2173125743865967
Dev set Loss 2.21502685546875
Dev set Loss 2.2127766609191895
Dev set Loss 2.210561513900757
Dev set Loss 2.2083797454833984
Dev set Loss 2.206230878829956
Dev set Loss 2.2041144371032715
Dev set Loss 2.20202898979187
Dev set Loss 2.199974298477173
Dev set Loss 2.197948694229126
Dev set Loss 2.195952892303467
Dev set Loss 2.1939852237701416
Dev set Loss 2.1920454502105713
Dev set Loss 2.1901328563690186
Dev set Loss 2.188246250152588
Dev set Loss 2.1863858699798584
Dev set Loss 2.1845507621765137
Dev set Loss 2.1827402114868164
Dev set Loss 2.1809544563293457
Dev set Loss 2.1791913509368896
Dev set Loss 2.1774520874023438
Dev set Loss 2.175734758377075
Dev set Loss 2.1740400791168213
Dev set Loss 2.1723673343658447
Dev set Loss 2.170715093612671
Dev set Loss 2.169083833694458
Dev set Loss 2.167473077774048
Dev set Loss 2.165881872177124
Dev set Loss 2.1643102169036865
Dev set Loss 2.162757635116577
Dev set Loss 2.1612236499786377
Dev set Loss 2.159707784

In [79]:
#### Test set loss
test_xs_encoded = F.one_hot(test_xs, num_classes=729).float()
cross_entropy_loss = F.cross_entropy(test_xs_encoded @ W, test_ys) + 0.007 * (W**2).mean()
print(f"Test set Loss {cross_entropy_loss}")

Test set Loss 2.187781810760498


In [81]:
# Name generation using trigram neural net
g = torch.Generator().manual_seed(2147481317)
trigram_names = []
for i in range(5):
  prev_char = '*'
  ix = trigram_idx_map['*' + prev_char]
  out = []
  while True:
    #x_encoded = F.one_hot(torch.tensor([ix]), num_classes=27).float()
    matmul = (W[ix]).reshape(1, -1)
    #print(matmul)
    #print(matmul.shape)
    counts = matmul.exp()
    probs = counts / counts.sum(1, keepdim=True)
    next_char_idx = torch.multinomial(probs, num_samples=1, replacement=True, generator=g).item()
    next_char = idx_to_char[next_char_idx]
    ix = trigram_idx_map[prev_char + next_char]
    prev_char = next_char
    out.append(next_char)
    if next_char == '*' or len(out) == 10:
      break
  trigram_names.append(''.join(out))

In [82]:
#### Interesting exercise
#### Creating a basic story with randomly generated character names using trigram neural net model


story_template = """{name1}, {name2}, {name3}, {name4}, and {name5} lived near an ancient forest.
One morning, {name2} found a golden key beside the river.
{name3} believed it opened a hidden door inside the forest.
{name1} packed some food, while {name4} brought a lantern.
{name5} carefully marked their path as they walked beneath the trees.
At sunset, they discovered a tiny door in an enormous oak tree.
{name2} used the golden key, and the door swung open.
Inside, the five friends found a garden filled with glowing flowers.
They promised to protect the magical garden and keep its location secret.
From that day onward, {name1}, {name2}, {name3}, {name4}, and {name5} visited it every full moon."""

story = story_template.format(
    name1=trigram_names[0],
    name2=trigram_names[1],
    name3=trigram_names[2],
    name4=trigram_names[3],
    name5=trigram_names[4],
)
print(story)

el*, lyn*, koldgevell, jer*, and iver* lived near an ancient forest.
One morning, lyn* found a golden key beside the river.
koldgevell believed it opened a hidden door inside the forest.
el* packed some food, while jer* brought a lantern.
iver* carefully marked their path as they walked beneath the trees.
At sunset, they discovered a tiny door in an enormous oak tree.
lyn* used the golden key, and the door swung open.
Inside, the five friends found a garden filled with glowing flowers.
They promised to protect the magical garden and keep its location secret.
From that day onward, el*, lyn*, koldgevell, jer*, and iver* visited it every full moon.
